In [1]:
import os, shutil, csv
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import CommitFailedError


Source_DIR = "/lakehouse/default/Files/source_csv"
Target_PATH = "abfss://tpch@onelake.dfs.fabric.microsoft.com/occ.Lakehouse/Tables/dbo/target_b"

In [2]:
os.makedirs(Source_DIR, exist_ok=True)
for batch_num in range(1, 6):
    fname = os.path.join(Source_DIR, f"batch_{batch_num:03d}.csv")
    start = (batch_num - 1) * 20 + 1
    with open(fname, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "value"])
        for i in range(start, start + 20):
            w.writerow([i, i * 10])

# Bootstrap B with just batch_001.csv ingested (filename tracks provenance)
write_deltalake(Target_PATH, pa.table({
    "id": pa.array(range(1, 21), pa.int64()),
    "value": pa.array([i * 10 for i in range(1, 21)], pa.int64()),
    "filename": pa.array(["batch_001.csv"] * 20),
}), mode="overwrite")

In [3]:
vB = DeltaTable(Target_PATH).version()
print(f"pinning B@v{vB}")

pinning B@v4


**Do Stuff with DuckDB, Pandas, Polars etc**

In [4]:
import duckdb
con = duckdb.connect()
# Read all CSVs, keep only rows from filenames not yet in B@vB.
our_rows = con.execute(f"""
    ATTACH '{Target_PATH}' AS tgt (TYPE delta, VERSION {vB});
    SELECT
        s.id,
        s.value,
        parse_filename(s.filename) AS filename
    FROM read_csv_auto('{Source_DIR}/*.csv', filename=true) s
    WHERE parse_filename(s.filename) NOT IN (SELECT DISTINCT filename FROM tgt)
    ORDER BY s.id
""").fetch_arrow_table()

new_files = sorted(set(our_rows.column("filename").to_pylist()))
print(f"new rows to ingest: {our_rows.num_rows} from files {new_files}")

new rows to ingest: 80 from files ['batch_002.csv', 'batch_003.csv', 'batch_004.csv', 'batch_005.csv']


In [5]:
DeltaTable(Target_PATH).delete("filename = 'batch_001.csv'")

{'num_added_files': 0,
 'num_removed_files': 1,
 'num_deleted_rows': 20,
 'num_copied_rows': 0,
 'execution_time_ms': 1007,
 'scan_time_ms': 807,
 'rewrite_time_ms': 200}

In [6]:
DeltaTable(Target_PATH).merge(
        source=our_rows,
        predicate="t.filename = s.filename",
        source_alias="s",
        target_alias="t",
    ).when_not_matched_insert_all().execute()

{'num_source_rows': 80,
 'num_target_rows_inserted': 80,
 'num_target_rows_updated': 0,
 'num_target_rows_deleted': 0,
 'num_target_rows_copied': 0,
 'num_output_rows': 80,
 'num_target_files_scanned': 0,
 'num_target_files_skipped_during_scan': 0,
 'num_target_files_added': 1,
 'num_target_files_removed': 0,
 'execution_time_ms': 527,
 'scan_time_ms': 217,
 'rewrite_time_ms': 0}

In [7]:
DeltaTable(Target_PATH).merge(
        source=our_rows,
        predicate="t.filename = s.filename",
        source_alias="s",
        target_alias="t",
    ).when_not_matched_insert_all().execute()

{'num_source_rows': 80,
 'num_target_rows_inserted': 0,
 'num_target_rows_updated': 0,
 'num_target_rows_deleted': 0,
 'num_target_rows_copied': 0,
 'num_output_rows': 0,
 'num_target_files_scanned': 1,
 'num_target_files_skipped_during_scan': 0,
 'num_target_files_added': 0,
 'num_target_files_removed': 0,
 'execution_time_ms': 53,
 'scan_time_ms': 41,
 'rewrite_time_ms': 0}

In [8]:
write_deltalake(Target_PATH,our_rows,mode='append')

In [9]:
try:
    DeltaTable(Target_PATH, version=vB).merge(
        source=our_rows,
        predicate="t.filename = s.filename",
        source_alias="s",
        target_alias="t",
    ).when_not_matched_insert_all().execute()
    print("FAIL: merge committed — pin did not detect the conflict")
except CommitFailedError as e:
    print(" ", e)

[2026-05-23T14:27:54Z WARN  deltalake_core::kernel::transaction] table updated during transaction, checking for conflicts base_version=4 latest_version=7 versions_behind=3
[2026-05-23T14:27:54Z ERROR deltalake_core::kernel::transaction] conflict detected, aborting transaction conflicts_checked=1 error=Commit failed: a concurrent transaction deleted data this operation read.
    Help: This transaction's query must be rerun to exclude the removed data. Also, if you don't care to require this check to pass in the future, the isolation level can be set to Snapshot Isolation.
